# Network ReAct Agent - Demo Notebook
"Reasoning Loop with External State"

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path so we can import the package
sys.path.insert(0, str(Path.cwd().parent / "src"))

import json

from langchain_core.messages import HumanMessage

from network_manager_agent import (
    LLMConfig,
    build_agent,
    create_llm,
)
from network_manager_agent.data import DataManager

## Load Data

In [ ]:
county_specialty_thresholds = {"mi": {"wayne": {"general practice": 10.0}}}

dm = DataManager(county_specialty_thresholds=county_specialty_thresholds)
candidates = dm.get_candidates_df()
members = dm.get_members_df()
entity_summaries = dm.get_entity_summaries()
schema_profile = json.dumps(dm.get_schema_profile(), indent=2)

print(f"Loaded {len(candidates)} candidates, {len(members)} members, {len(entity_summaries)} entities")

## Pre-compute: Filter & Aggregate (done by DataManager)

In [ ]:
# DataManager handles loading, filtering, aggregation, and schema profiling automatically.
# See the Setup cell above for the current configuration.
print(f"Candidates: {len(dm.get_candidates_df())}, Members: {len(dm.get_members_df())}, Entities: {len(dm.get_entity_summaries())}")

## Create LLM

In [ ]:
llm_config = LLMConfig()
llm = create_llm(llm_config)

# Quick test
response = llm.invoke("Hey there! Just making sure you're up and running!")
response.pretty_print()

## Build Agent

In [ ]:
import sqlite3
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

db_path = str(Path.cwd().parent / "checkpoints.sqlite")
conn = sqlite3.connect(db_path, check_same_thread=False)
checkpointer = SqliteSaver(conn)
agent = build_agent(llm, checkpointer=checkpointer)
print("Agent built successfully!")


## Visualize Graph

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))


## Run the Agent

In [ ]:
from network_manager_agent.ui import run_agent


def run_agent_demo(agent, inputs: dict, config: dict):
    """Wrapper for run_agent that matches the notebook's naming convention."""
    run_agent(agent, inputs, config)


In [ ]:
# Create a thread
config = {"configurable": {"thread_id": "1"}}


In [ ]:
# Specify an input
messages = [HumanMessage(content='''
When constructing or analyzing how to build a network, what are some of the patterns you should use when running code?
''')]

inputs = {
    "messages": messages,
    "county_specialty_thresholds": county_specialty_thresholds,
    "entity_summaries": entity_summaries,
    "schema_profile": schema_profile,
}

run_agent_demo(agent, inputs, config)


In [ ]:
# Create a thread
config = {"configurable": {"thread_id": "1"}}

# Specify an input
messages = [HumanMessage(content='''
I'd like to find a good contract entity to add to the network. I want it to push my coverage as high as possible. What recommendations do you have?
''')]

inputs = {
    "messages": messages,
    "county_specialty_thresholds": county_specialty_thresholds,
    "entity_summaries": entity_summaries,
    "schema_profile": schema_profile,
}

run_agent_demo(agent, inputs, config)


## Inspect Agent State

In [ ]:
# Show the last human message from the conversation
state = agent.get_state(config)
messages = state.values.get("messages", [])
last_human = [m for m in reversed(messages) if m.type == 'human']
print("Last human message:")
print(last_human[0].content if last_human else "(none)")


In [ ]:
print("Summary:")
print(state.values.get("summary", "(empty)"))
print()

print("Entity summaries in state:")
print(f"  {len(state.values.get('entity_summaries', []))} entities loaded")
print()

print("Schema profile:")
schema = state.values.get("schema_profile", "")
print(f"  {len(schema)} chars" if schema else "  (empty)")


In [ ]:
import pandas as pd
print("Current network (list of entity IDs):")
network = agent.get_state(config).values.get("network", [])
if network:
    for i, entity_id in enumerate(network, 1):
        print(f"  {i}. {entity_id}")
else:
    print("  (empty - no entities added yet)")


## Try a More Complex Prompt

In [ ]:
# Reset with a new thread
# config = {"configurable": {"thread_id": "2"}}

messages = [HumanMessage(content='''
OK, now I'd like to add one more contract entity to the network. But I'd like to make sure this one has high effectiveness, but also one that best adds to the coverage. Find one, add it, and then report back on the status of the network.
''')]

inputs = {
    "messages": messages,
    "county_specialty_thresholds": county_specialty_thresholds,
    "entity_summaries": entity_summaries,
    "schema_profile": schema_profile,
}

run_agent_demo(agent, inputs, config)


In [ ]:
# Check final state
state = agent.get_state(config)
print("Summary:")
print(state.values.get("summary", "(empty)"))

print("\nNetwork (entity IDs):")
network = state.values.get("network", [])
if network:
    for i, entity_id in enumerate(network, 1):
        print(f"  {i}. {entity_id}")
else:
    print("  (empty)")
